<a href="https://colab.research.google.com/github/CesarChaMal/text-generation-webui/blob/main/Colab-MultiModel-TextGen-GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# oobabooga/text-generation-webui

After running both cells, a public gradio URL will appear at the bottom in a few minutes. You can optionally generate an API link.

* Project page: https://github.com/oobabooga/text-generation-webui
* Gradio server status: https://status.gradio.app/

In [ ]:
#@title 1. Keep this tab alive to prevent Colab from disconnecting you { display-mode: "form" }

#@markdown Press play on the music player that will appear below:
%%html
<audio src="https://oobabooga.github.io/silence.m4a" controls>

In [ ]:
!rm -rf *
#@title 2. Launch the web UI with multi model option
import os
import subprocess
from pathlib import Path

# Function to download a model
def download_model(model_url, branch, model_folder):
    url_parts = model_url.strip('/').split('/')
    output_folder = f"{model_folder}/{url_parts[-2]}_{url_parts[-1]}"
    if branch not in ['', 'main']:
        output_folder += f"_{branch}"
    subprocess.run(["python", "download-model.py", model_url, "--branch", branch])
    return output_folder

# Ensure the current directory is correct
if Path.cwd().name != 'text-generation-webui':
    print("Installing the webui...")
    #!git clone https://github.com/oobabooga/text-generation-webui
    !git clone https://github.com/CesarChaMal/text-generation-webui.git
    %cd text-generation-webui

#@markdown If unsure about the branch, write "main" or leave it blank.
# List of models to download
models = [
    {"url": "https://huggingface.co/TheBloke/MythoMax-L2-13B-GPTQ", "branch": "gptq-4bit-32g-actorder_True"},
    {"url": "https://huggingface.co/TheBloke/Wizard-Vicuna-13B-Uncensored-GPTQ", "branch": "gptq-4bit-32g-actorder_True"},
    {"url": "https://huggingface.co/TheBloke/WizardLM-13B-V1.0-Uncensored-GPTQ", "branch": "gptq-4bit-32g-actorder_True"},
    {"url": "https://huggingface.co/TheBloke/Guanaco-13B-Uncensored-GPTQ", "branch": "gptq-4bit-32g-actorder_True"},
    {"url": "https://huggingface.co/TheBloke/Uncensored-Jordan-13B-GPTQ", "branch": "gptq-4bit-32g-actorder_True"},
    {"url": "https://huggingface.co/TheBloke/Uncensored-Frank-13b-GPTQ", "branch": "gptq-4bit-32g-actorder_True"},
    {"url": "https://huggingface.co/teknium/OpenHermes-13B", "branch": "main"},
    {"url": "https://huggingface.co/deepseek-ai/deepseek-llm-7b-chat", "branch": "main"},
]

# Create a directory for models if it doesn't exist
os.makedirs("models", exist_ok=True)

# Download models
model_folders = []
for model in models:
    folder = download_model(model["url"], model["branch"], "models")
    if folder:
        model_folders.append(folder)

# Set up the environment and install necessary packages
torver = torch.__version__
print(f"TORCH: {torver}")

textgen_requirements = open('requirements.txt').read().splitlines()
with open('temp_requirements.txt', 'w') as file:
    file.write('\n'.join(textgen_requirements))

!pip install -r extensions/openai/requirements.txt --upgrade
!pip install -r temp_requirements.txt --upgrade

try:
    import flash_attn
except:
    !pip uninstall -y flash_attn

print("\n --> If you see a warning about 'previously imported packages', just ignore it.")
print("\n --> There is no need to restart the runtime.")

# Parameters for the web UI
command_line_flags = "--n-gpu-layers 128 --load-in-4bit --use_double_quant"  # Modify as needed
api = False  # Modify as needed (if you need API support)

# Launch the web UI with a selected model
selected_model = model_folders[0]  # Modify index based on user selection
cmd = f"python server.py --share --model {selected_model} {command_line_flags}"
print(f"Executing command: {cmd}")
subprocess.run(cmd.split())
